In [2]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        print("입력 Shape:", src.size())

        embedded = self.embedding(src)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        outputs, (h_0, c_0) = self.rnn(embedded)
        print("LSTM Layer의 Output Shape:", outputs.size())
        print("LSTM Layer의 Hidden State Shape:", h_0.size())
        print("LSTM Layer의 Cell State Shape:", c_0.size())

        return outputs, h_0, c_0

In [3]:
vocab_size = 30000
emb_size = 256
lstm_size = 512
batch_size = 1
sample_seq_len = 3

print("Vocab Size: {0}".format(vocab_size))
print("Embedidng Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))

Vocab Size: 30000
Embedidng Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



In [3]:
import torch

encoder = Encoder(vocab_size, emb_size, lstm_size)
sample_input = torch.randint(0, vocab_size, (batch_size, sample_seq_len))

sample_output, hidden, cell = encoder(sample_input)

입력 Shape: torch.Size([1, 3])
Embedding Layer를 거친 Shape: torch.Size([1, 3, 256])
LSTM Layer의 Output Shape: torch.Size([1, 3, 512])
LSTM Layer의 Hidden State Shape: torch.Size([1, 1, 512])
LSTM Layer의 Cell State Shape: torch.Size([1, 1, 512])


![대체 텍스트](다운로드 (1).png)

In [4]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Decoder, self).__init__()
        # 단어 인덱스를 embedding_dim 크기의 벡터로 변환
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # LSTM 레이어 설정: 입력 크기는 (현재 단어 임베딩 + 인코더의 컨텍스트 벡터)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, batch_first=True)
        
        # LSTM 출력을 기반으로 다음에 올 단어(전체 vocab_size 중 하나)를 예측
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden, cell, context):
        # x: 현재 입력 단어 인덱스, context: 인코더의 요약 정보(컨텍스트 벡터)
        print("입력 Shape:", x.size())

        # 1. 단어를 임베딩 벡터로 변환
        embedded = self.embedding(x)
        print("Embedding Layer를 거친 Shape:", embedded.size())

        # 2. 임베딩된 단어 벡터와 인코더의 컨텍스트 벡터를 하나로 결합 (Concatenation)
        # dim=2는 피처(feature) 차원을 따라 결합함을 의미함
        embedded = torch.cat((embedded, context), dim=2)
        print("Context Vector가 더해진 Shape:", embedded.size())

        # 3. 결합된 벡터를 LSTM에 입력 (이전 단계의 hidden, cell 상태도 함께 전달)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        print("LSTM Layer의 Output Shape:", output.size())

        # 4. 최종 출력 레이어를 통해 다음에 올 단어 예측
        output = self.fc(output)
        print("Decoder 최종 Output Shape:", output.size())

        return output, hidden, cell

print("Vocab Size: {0}".format(vocab_size))
print("Embedidng Size: {0}".format(emb_size))
print("LSTM Size: {0}".format(lstm_size))
print("Batch Size: {0}".format(batch_size))
print("Sample Sequence Length: {0}\n".format(sample_seq_len))      

Vocab Size: 30000
Embedidng Size: 256
LSTM Size: 512
Batch Size: 1
Sample Sequence Length: 3



![대체 텍스트](다운로드.png)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BahdanauAttention(nn.Module):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W_decoder = nn.Linear(512, units)  # Decoder hidden state -> units
        self.W_encoder = nn.Linear(512, units)  # Encoder hidden state -> units
        self.W_combine = nn.Linear(units, 1)   # Alignment score -> scalar weight

    # 어텐션 계산 과정
    def forward(self, H_encoder, H_decoder):
        print("[ H_encoder ] Shape:", H_encoder.shape)  # (batch, seq_len, hidden_dim)

        H_encoder = self.W_encoder(H_encoder)
        print("[ W_encoder X H_encoder ] Shape:", H_encoder.shape)  # (batch, seq_len, units)

        print("\n[ H_decoder ] Shape:", H_decoder.shape)  # (batch, hidden_dim)
        H_decoder = H_decoder.unsqueeze(1)  # (batch, 1, hidden_dim)
        H_decoder = self.W_decoder(H_decoder)  # (batch, 1, units)

        print("[ W_decoder X H_decoder ] Shape:", H_decoder.shape)  # (batch, 1, units)

        # Additive(덧셈) 어텐션
        score = self.W_combine(torch.tanh(H_decoder + H_encoder))  # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", score.shape)

        #  Attention Distribution
        attention_weights = F.softmax(score, dim=1)  # (batch, seq_len, 1)
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())


        # atten value
        context_vector = attention_weights * H_decoder  # (batch, seq_len, units)
        context_vector = torch.sum(context_vector, dim=1)  # (batch, units)

        return context_vector, attention_weights

# 설정
W_size = 100
print(f"Hidden State를 {W_size}차원으로 Mapping\n")

# 모델 생성
attention = BahdanauAttention(W_size)

# 입력 데이터 (배치 크기 = 1)
enc_state = torch.rand((1, 10, 512))  # (batch, seq_len, hidden_dim)
dec_state = torch.rand((1, 512))  # (batch, hidden_dim)

# 실행
_ = attention(enc_state, dec_state)

Hidden State를 100차원으로 Mapping

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 100])

[ H_decoder ] Shape: torch.Size([1, 512])
[ W_decoder X H_decoder ] Shape: torch.Size([1, 1, 100])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.09238198 0.10175455 0.10383356 0.10297031 0.09523237 0.09834632
  0.09760129 0.0935526  0.09851226 0.11581479]]


In [ ]:
class LuongAttention(nn.Module):
    def __init__(self, units):
        super(LuongAttention, self).__init__()
        self.W_combine = nn.Linear(units, units)  # Encoder hidden state 변환

    def forward(self, H_encoder, H_decoder):
        print("[ H_encoder ] Shape:", H_encoder.shape)  # (batch, seq_len, hidden_dim)

        WH = self.W_combine(H_encoder)  # (batch, seq_len, hidden_dim)
        print("[ W_encoder X H_encoder ] Shape:", WH.shape)

        H_decoder = H_decoder.unsqueeze(1)  # (batch, 1, hidden_dim)
        # 곱셈/내적(Dot-product) 방식
        alignment = torch.bmm(WH, H_decoder.transpose(1, 2))  # (batch, seq_len, 1)
        print("[ Score_alignment ] Shape:", alignment.shape)

        attention_weights = F.softmax(alignment, dim=1)  # (batch, seq_len, 1)
        print("\n최종 Weight:\n", attention_weights.squeeze(-1).detach().numpy())

        attention_weights = attention_weights.squeeze(-1)  # (batch, seq_len)
        context_vector = torch.bmm(attention_weights.unsqueeze(1), H_encoder)  # (batch, 1, hidden_dim)
        context_vector = context_vector.squeeze(1)  # (batch, hidden_dim)

        return context_vector, attention_weights

# 설정
emb_dim = 512
attention = LuongAttention(emb_dim)

# 입력 데이터 (배치 크기 = 1)
enc_state = torch.rand((1, 10, emb_dim))  # (batch, seq_len, hidden_dim)
dec_state = torch.rand((1, emb_dim))  # (batch, hidden_dim)

# 실행
_ = attention(enc_state, dec_state)

[ H_encoder ] Shape: torch.Size([1, 10, 512])
[ W_encoder X H_encoder ] Shape: torch.Size([1, 10, 512])
[ Score_alignment ] Shape: torch.Size([1, 10, 1])

최종 Weight:
 [[0.00886931 0.01847257 0.01439751 0.01096491 0.10831723 0.17651267
  0.5479286  0.06558592 0.02304609 0.02590528]]
